In [ ]:
import pygmt
import pandas as pd
import geopandas as gpd
# import warnings
# warnings.filterwarnings('ignore')
from mat73 import loadmat
import numpy as np
from scipy.interpolate import RegularGridInterpolator
import os
os.chdir('../..')
import netCDF4 as nc
import xarray as xr
from pygmt.params import Box, Position

import pyproj
pyproj.datadir.get_data_dir()
# setting proj to the proj within this environment
pyproj.datadir.set_data_dir('/opt/homebrew/Caskroom/miniforge/base/envs/pygmt/share/proj')

In [ ]:
# ### creating scaffold of figure first
# fig = pygmt.Figure()
# w  = 10
# h  = 3

# ### Bottom row, not as tall
# with fig.subplot(nrows=1, ncols=1, figsize=(f"{w}c", f"{h}c"), autolabel="b)"):
#     fig.basemap(
#         region=[0, 5, 0, 5], projection="X?", frame=["af", "WSne"], panel=[0, 0]
#     )
    
# ### Move plot origin by 1 cm above the height of the entire figure
# fig.shift_origin(yshift="h+1c")
# ### Top row, rectangle
# with fig.subplot(nrows=1, ncols=1, figsize=(f"{w}c", f"{w}c"), autolabel="a)"):
#     fig.basemap(
#         region=[0, 10, 0, 10], projection="X?", frame=["af", "WSne"], panel=[0, 0]
#     )

# fig.show()

In [ ]:
### read in ice history
iceHist = xr.open_dataset('inputs/iceHistories/iceHistory-CISLGM_icepc_022226_highres_256.nc')
### need to solvefor ind
grid_ice = iceHist["ice_thickness"].sel(time=17)

In [ ]:
### read in Gebco
grid_GEBCO = xr.open_dataset('inputs/GEBCO_03_Nov_2022_a5af2d998654/gebco_2022_n67.0_s43.0_w-179.0_e-117.0.nc')
grid_GEBCO = grid_GEBCO['elevation']

# grid_GEBCO = 
# nc.Dataset('inputs/GEBCO_03_Nov_2022_a5af2d998654/gebco_2022_n67.0_s43.0_w-179.0_e-117.0.nc')
# GEBCO_lat = GEBCO['lat'][:]
# GEBCO_lon = GEBCO['lon'][:]
# GEBCO_elev = GEBCO['elevation'][:,:]

# GEBCO_LON,GEBCO_LAT = np.meshgrid(GEBCO_lon,GEBCO_lat)

In [ ]:
### read in rsl of ice history
rsl = nc.Dataset('inputs/rsl/rsl-CISLGM_icepc_022226_highres-prem.l50.ump2.lm3_lvz_200km.yt.nc')

In [ ]:
# grid_gebco = pygmt.xyz2grd(
#     x = GEBCO_LON.flatten(),
#     y = GEBCO_LAT.flatten(),
#     z = GEBCO_elev.flatten(),
#     region = [float(np.min(GEBCO_lon)), float(np.max(GEBCO_lon)), float(np.min(GEBCO_lat)), float(np.max(GEBCO_lat))],
#     # spacing = (f'{GEBCO_lon.shape[0]}+n',f'{GEBCO_lat.shape[0]}+n')
#     spacing = (0.00416667,0.00416667)
# )

In [ ]:
vols_out = np.loadtxt('outputs/iceVolTime.txt')

In [ ]:
### read in flowlines
jdf = pd.read_excel('inputs/flowlines/jdf_update.xlsx')
sv = pd.read_excel('inputs/flowlines/sv.xlsx')
ysv = pd.read_excel('inputs/flowlines/ysv_update.xlsx')
flowlines = [jdf, sv, ysv]

In [ ]:
clr_n = '#75DDDD'
clr_c = '#9297C4'
clr_s = '#AA3E98'

In [ ]:
### creating figure
fig = pygmt.Figure()
### set up for figure
w  = 10
h  = 5
### creating region and extents
xmin = 360 - 146.
xmax = 360 - 120.
ymin = 44.
ymax = 62.
region = [xmin,xmax,ymin,ymax]
projection = f'M0/0/{w}c'
###

####################
### left panel, map of ice extent
###################
with fig.subplot(nrows=1, ncols=1, figsize=(f"{w}c", f"{w}c"), autolabel="a)"):
    ### creating basemap
    fig.basemap(region = region,
         projection = projection,
               frame = ['afg'])

    ### plotting topography
    pygmt.makecpt(cmap="gmt/globe+h", series=[-8000, 2000])
    fig.grdimage(grid=grid_GEBCO,
                  region = region,
                  projection = projection,
                 )
    
    fig.colorbar(frame = ['a2000f500', "x+lElevation (m)"], position=Position("LM", 'outside', offset = (-2,0.5), anchor = 'CM'), 
                 orientation = 'vertical', 
                 move_text = ['annotations', 'label', 'unit'])

    ### boundaries and shorelines
    fig.coast(shorelines="1/0.5p,black", borders = '2/0.5p,black', region = region, projection = projection)

    ### plotting ice extent at 17 ka
    # pygmt.makecpt(cmap="gmt/globe", series=[-6000, 0])
    # fig.grdimage(grid=grid_ice,
    #          region = region,
    #      projection = projection,
    #      nan_transparent = True, 
    #      transparency = 50,
    #             )

    x2, y2 = np.meshgrid(grid_ice['lon'].values, grid_ice['lat'].values)
    dum = np.copy(np.ravel(grid_ice.values))
    dum[np.isnan(dum)] = 0
    fig.contour(region = region,
                projection = projection,
                x = np.ravel(x2),
                y = np.ravel(y2),
                z = dum,
                levels = [1],
                pen = '2p,white',
               )
    
    for flow in flowlines:
        fig.plot(x = flow['Lon'], y = flow['Lat'], pen = '2p,black', projection = projection, region=region)

    ### Creating inset
    xmin_inset, xmax_inset = -180, -40
    ymin_inset, ymax_inset = 30, 70
    region_inset = [xmin_inset, xmax_inset, ymin_inset, ymax_inset]
    with fig.inset(
        position=Position("BL", offset=0.1),
        # box=Box(fill="white", pen="1p"),
        region=region_inset,
        projection=f"L{(-100)}/35/{ymin_inset}/{ymax_inset}/6c",
    ):
        with pygmt.config(
            MAP_FRAME_TYPE="plain",
            MAP_FRAME_PEN="1p,black",
            # MAP_TICK_PEN="0p",
            # FONT_ANNOT_PRIMARY="0p",
        ):
            ### adding coastlinies
            fig.coast(shorelines="1/0.5p,black", region = region_inset, land="gray70", water="white",frame = ['wnse'])

        ### plotting rectangle over area
        rectangle = [[region[0], region[2], region[1], region[3]]]
        fig.plot(data=rectangle, style="r+s", pen="1p,black")
        


fig.shift_origin(xshift=f"w+{2.5}c")

####################
### bottom right row, ice growth and gmsl
###################
with fig.subplot(nrows=1, ncols=1, figsize=(f"{w}c", f"{h}c"), autolabel="b)"):
    fig.basemap(
        region=[0, 40, 0, 2.1e15], 
        projection="X?", frame=["af", "WSe", "xaf+lTime (ka)", "yaf+lIce Volume (m@+3@+)"], 
        panel=[0, 0],
        
    )

    fig.plot(x = vols_out[:,0], y=vols_out[:,1], label = "North", pen = f'1p,{clr_n}')
    fig.plot(x = vols_out[:,0], y=vols_out[:,2], label = "Central", pen = f'1p,{clr_c}')
    fig.plot(x = vols_out[:,0], y=vols_out[:,3], label = "South", pen = f'1p,{clr_s}')

    fig.plot(x=34, y=0, style = 't0.4c', pen = '1p,black', fill = 'black')
    fig.plot(x=17, y=0, style = 't0.4c', pen = '1p,black', fill = 'black')

    fig.basemap(region=[0, 40, -140, 4], frame=["E", "yaf+lGlobal Mean Sea Level (m)"])

    fig.plot(x = rsl['t'][:], y = rsl['gmsl'][:], label = 'Global Mean', pen = '1p,black')
    # text for legend
    fig.plot(
        x=[None],
        y=[None],
        label="Sea Level"
    )

   
    fig.legend(position=Position("BL",offset = 0.2),box=True, scale = 1)
    # fig.legend(position="BL+o0.2c/0.2c+w4c",box = True)


####################
### top right row, times of ice growth
###################
fig.shift_origin(yshift=f"h+0.1c")
with fig.subplot(nrows=1, ncols=1, figsize=(f"{w}c", f"{1}c")):
    fig.basemap(
            region=[0, 40, 1.6e15, 2.1e15], 
            projection="X?", frame=['n'], 
            panel=[0, 0],
            )
    lw = 2.5
    ### plotting lines of times of ice sheet growth
    fig.plot(x = [35,20], y = [1.9e15,1.9e15], pen = f'{lw}p,{clr_n}')
    fig.plot(x = [30,20], y = [1.8e15,1.8e15], pen = f'{lw}p,{clr_c}')
    fig.plot(x = [35,17], y = [1.7e15,1.7e15], pen = f'{lw}p,{clr_s}')


fig.show()
# fig.savefig('figures/f01_transectMap.pdf')